🟢 Débutant | ⏱ 45 min | 🔑 lire, nettoyer, écrire

# Module 08 — Fil rouge MétéoLocale : le vrai fichier

**Lundi matin chez MétéoLocale.** Inès pose deux fichiers sur ton bureau : « Voici les **vrais** relevés de
septembre, `meteo.csv`, 180 lignes. Et voici `meteo_brut.csv` : la même chose saisie par l'ancien stagiaire...
virgules, `N/A`, doublons, je te laisse découvrir. Il me faut pour midi : le fichier propre, et un **résumé par
ville en JSON** que Karim chargera sur le site. »

**Ce qu'on construit aujourd'hui** : un petit **pipeline** (une chaîne d'étapes, comme une chaîne de montage) :
fichier brut → nettoyage → fichier propre ; fichier propre → résumés → JSON.

**Ce qu'on a construit avant** :
- 01 à 03 : la fiche d'une journée, mise en forme et classée (pluie, vent, chaleur) ;
- 04 et 05 : les 30 jours parcourus avec des boucles, les statistiques de Toulouse dans une liste ;
- 06 : le jeu de données en liste de dictionnaires (écrit à la main) ;
- 07 : la boîte à outils de fonctions `moyenne`, `celsius_vers_fahrenheit`, `resume_ville`.

Les étapes s'enchaînent : exécute-les dans l'ordre. Commence par la cellule ci-dessous, qui contient tes outils du
module 07 (recopiés : au module 09, tu apprendras à les ranger dans ton propre module pour ne plus les recopier).

### Lire un échec de vérification

Quand une vérification échoue, Python affiche un long message rouge, surtout en anglais. **Ne lis que la
dernière ligne** :

```
AssertionError: moyenne doit valoir 21.5, tu as obtenu 25.0 : as-tu divisé par le nombre de jours ?
```

- `AssertionError` veut dire « **la vérification a échoué** » (ton code tourne, mais le résultat n'est pas encore le bon) ;
- le **message en français**, après les deux-points, te dit **quoi corriger** ;
- tout ce qui est au-dessus (en anglais, avec des numéros de ligne) peut être **ignoré**.

Si la dernière ligne commence par un autre mot (`NameError`, `TypeError`...), c'est **ta cellule** qui a un
problème : relis la section ⚠️ du cours qui parle de cette erreur.

Les caractères spéciaux (émojis comme ☀️ 🌧️ 💨, flèche →, symbole °) ne se tapent pas facilement : **copie-colle-les
depuis l'énoncé**.

In [ ]:
import csv
import json
from pathlib import Path

DONNEES = Path("../donnees")
SORTIES = Path("sorties")
SORTIES.mkdir(exist_ok=True)
COLONNES = ["date", "ville", "temp_min", "temp_max", "pluie_mm", "vent_kmh"]

# --- La boîte à outils du module 07 ---
def moyenne(valeurs):
    """Renvoie la moyenne d'une liste non vide de nombres."""
    return sum(valeurs) / len(valeurs)

def celsius_vers_fahrenheit(celsius, chiffres=1):
    """Convertit des °C en °F, arrondi à `chiffres` chiffres."""
    return round(celsius * 9 / 5 + 32, chiffres)

def valeurs_ville(releves, ville, cle):
    """Liste des valeurs de la colonne cle pour la ville demandée (les None sont ignorés)."""
    resultat = []
    for releve in releves:
        if releve["ville"] == ville and releve[cle] is not None:
            resultat.append(releve[cle])
    return resultat

print("Outils prêts.")

### Étape 1 — Charger le vrai fichier

**Objectif** : Écrire `charger_meteo(chemin)` qui renvoie les 180 relevés avec de vrais nombres.

**À faire** :
1. Lis le CSV avec `csv.DictReader`.
2. Pour chaque ligne, construis un relevé : `date` et `ville` en texte, `temp_min`, `temp_max`, `pluie_mm` en `float`, `vent_kmh` en `int`.
3. Range dans `releves` le résultat pour `DONNEES / "meteo.csv"`.

**Résultat attendu** :
```
180 relevés
{'date': '2026-09-01', 'ville': 'Toulouse', 'temp_min': 12.0, 'temp_max': 25.9, 'pluie_mm': 17.4, 'vent_kmh': 11}
```

<details><summary>💡 Indice</summary>

`float(ligne["temp_min"])`, etc. ; `int(ligne["vent_kmh"])`.

📖 Cours, section 6 (fonction `charger_meteo`).

</details>

In [ ]:
def charger_meteo(chemin):
    """Lit un CSV MétéoLocale propre et renvoie la liste des relevés convertis."""
    releves = []
    with open(chemin, encoding="utf-8", newline="") as fichier:
        for ligne in csv.DictReader(fichier):
            releve = {
                "date": ligne["date"],
                "ville": ligne["ville"],
                "temp_min": ...,
                "temp_max": ...,
                "pluie_mm": ...,
                "vent_kmh": ...,
            }
            releves.append(releve)
    return releves

releves = charger_meteo(DONNEES / "meteo.csv")
print(len(releves), "relevés")
print(releves[0])

In [ ]:
# Vérification
assert "releves" in dir() and len(releves) == 180, "releves doit contenir les 180 relevés de meteo.csv (exécute d'abord la cellule des outils)."
assert releves[0]["temp_max"] == 25.9 and releves[0]["pluie_mm"] == 17.4, f"Les nombres doivent être convertis avec float : tu as {releves[0]}."
assert releves[0]["vent_kmh"] == 11, f"vent_kmh doit être l'entier 11, tu as {releves[0]['vent_kmh']!r} : int(...)."
print("✅ OK")

### Étape 2 — Le résumé d'une ville

**Objectif** : Réutiliser tes outils du module 07 sur les 180 vrais relevés.

**À faire** :
1. Écris `resume_ville(releves, ville)` qui renvoie le dictionnaire :
2. `ville`, `nb_jours` (nombre de temp_max connues), `tmax_moyenne` (arrondie à 1 chiffre), `tmax_moyenne_f` (en °F), `pluie_totale` (arrondie à 1 chiffre).
3. Range dans `resume_toulouse` le résumé de Toulouse.

**Résultat attendu** :
```
{'ville': 'Toulouse', 'nb_jours': 30, 'tmax_moyenne': 23.9, 'tmax_moyenne_f': 75.0, 'pluie_totale': 108.3}
```

<details><summary>💡 Indice</summary>

C'est **exactement** l'étape 4 du fil rouge du module 07 : `valeurs_ville`, `moyenne`, `celsius_vers_fahrenheit`.

📖 Module 07, fil rouge, étape 4.

</details>

In [ ]:
def resume_ville(releves, ville):
    """Résumé d'une ville : nombre de jours, temp_max moyenne (°C et °F), pluie totale."""
    ...

resume_toulouse = resume_ville(releves, "Toulouse")
print(resume_toulouse)

In [ ]:
# Vérification
assert "resume_toulouse" in dir() and type(resume_toulouse) == dict, "resume_toulouse doit être un dictionnaire : as-tu écrit return ?"
attendu = {"ville": "Toulouse", "nb_jours": 30, "tmax_moyenne": 23.9, "tmax_moyenne_f": 75.0, "pluie_totale": 108.3}
for cle in attendu:
    assert resume_toulouse.get(cle) == attendu[cle], f"La clé {cle!r} doit valoir {attendu[cle]!r}, tu as {resume_toulouse.get(cle)!r}."
print("✅ OK")

### Étape 3 — Nettoyer le fichier brut

**Objectif** : Transformer `meteo_brut.csv` en 180 relevés propres.

**À faire** :
1. Complète `convertir_nombre` (virgule → point, `None` si impossible) et `nettoyer_ville`.
2. La boucle lit le brut (séparateur `;`), nettoie chaque ligne et ignore les doublons de `(date, ville)`.
3. Range le résultat dans `propres`.

**Résultat attendu** :
```
185 lignes brutes -> 180 relevés propres
```

<details><summary>💡 Indice</summary>

`float(texte.replace(",", "."))` ; `None` ; `texte[0].upper() + texte[1:].lower()`.

📖 Cours, sections 9 et 10.

</details>

In [ ]:
def convertir_nombre(texte):
    """'25,9' -> 25.9 ; '' ou 'N/A' -> None."""
    try:
        return ...
    except ValueError:
        return ...

def nettoyer_ville(texte):
    """'  montpellier ' -> 'Montpellier'."""
    texte = texte.strip()
    return ...

propres = []
deja_vus = set()
nb_lignes = 0
with open(DONNEES / "meteo_brut.csv", encoding="utf-8", newline="") as fichier:
    for ligne in csv.DictReader(fichier, delimiter=";"):
        nb_lignes = nb_lignes + 1
        releve = {"date": ligne["date"], "ville": nettoyer_ville(ligne["ville"])}
        for col in ["temp_min", "temp_max", "pluie_mm", "vent_kmh"]:
            releve[col] = convertir_nombre(ligne[col])
        cle = (releve["date"], releve["ville"])
        if cle not in deja_vus:
            deja_vus.add(cle)
            propres.append(releve)

print(nb_lignes, "lignes brutes ->", len(propres), "relevés propres")

In [ ]:
# Vérification
assert convertir_nombre("23,9") == 23.9 and convertir_nombre("N/A") is None, "convertir_nombre doit transformer '23,9' en 23.9 et 'N/A' en None."
assert nettoyer_ville("  tarbes ") == "Tarbes", f"nettoyer_ville('  tarbes ') doit renvoyer 'Tarbes', tu obtiens {nettoyer_ville('  tarbes ')!r}."
assert len(propres) == 180, f"propres doit contenir 180 relevés, tu en as {len(propres)} : vérifie le nettoyage des villes (sinon les doublons ne sont pas reconnus)."
assert set(r["ville"] for r in propres) == {"Toulouse", "Montpellier", "Nîmes", "Perpignan", "Albi", "Tarbes"}, "propres doit contenir exactement les 6 villes bien écrites."
print("✅ OK")

### Étape 4 — Écrire le fichier propre

**Objectif** : Sauvegarder les relevés nettoyés dans `sorties/meteo_propre.csv`.

**À faire** :
1. Écris `propres` dans `SORTIES / "meteo_propre.csv"` avec `csv.DictWriter`, les colonnes `COLONNES`, en-tête compris.
2. Range dans `nb_relus` le nombre de relevés relus dans ce fichier.

**Résultat attendu** :
```
180 relevés relus dans sorties/meteo_propre.csv
```

<details><summary>💡 Indice</summary>

Trois lignes dans le `with` : `ecrivain = csv.DictWriter(fichier, fieldnames=COLONNES)`, `ecrivain.writeheader()`, `ecrivain.writerows(propres)`.

📖 Cours, section 7.

</details>

In [ ]:
chemin_propre = SORTIES / "meteo_propre.csv"
with open(chemin_propre, "w", encoding="utf-8", newline="") as fichier:
    ...

with open(chemin_propre, encoding="utf-8", newline="") as fichier:
    nb_relus = len(list(csv.DictReader(fichier)))
print(nb_relus, "relevés relus dans", chemin_propre)

In [ ]:
# Vérification
assert nb_relus == 180, f"Le fichier propre doit contenir 180 relevés, il en contient {nb_relus} : as-tu appelé writeheader() puis writerows(propres) ?"
print("✅ OK")

### Étape 5 — Le résumé JSON pour le site

**Objectif** : Calculer le résumé des 6 villes et l'écrire en JSON.

**À faire** :
1. Construis le dictionnaire `bulletin` : `{"mois": "2026-09", "source": "meteo.csv", "villes": [...]}` où `villes` est la liste des 6 résumés (avec `resume_ville`), dans l'ordre de `villes`.
2. Écris-le dans `SORTIES / "resume_septembre.json"` (`ensure_ascii=False, indent=2`).

**Résultat attendu** :
```
Résumé écrit : 6 villes
```

<details><summary>💡 Indice</summary>

Dans la boucle : `bulletin["villes"].append(resume_ville(releves, ville))`. Puis `json.dump(bulletin, fichier, ensure_ascii=False, indent=2)`.

📖 Cours, section 8.

</details>

In [ ]:
villes = ["Toulouse", "Montpellier", "Nîmes", "Perpignan", "Albi", "Tarbes"]
bulletin = {"mois": "2026-09", "source": "meteo.csv", "villes": []}
for ville in villes:
    ...

with open(SORTIES / "resume_septembre.json", "w", encoding="utf-8") as fichier:
    ...
print("Résumé écrit :", len(bulletin["villes"]), "villes")

In [ ]:
# Vérification
assert len(bulletin["villes"]) == 6, f"bulletin['villes'] doit contenir 6 résumés, il en contient {len(bulletin['villes'])}."
assert bulletin["villes"][2]["ville"] == "Nîmes" and bulletin["villes"][2]["tmax_moyenne"] == 26.7, "Le 3e résumé doit être celui de Nîmes (tmax_moyenne 26.7)."
with open(SORTIES / "resume_septembre.json", encoding="utf-8") as f:
    texte = f.read()
assert "Nîmes" in texte, "Le fichier JSON doit contenir 'Nîmes' en clair (ensure_ascii=False) : as-tu appelé json.dump ?"
print("✅ OK")

### Étape 6 — Relire le JSON, prudemment

**Objectif** : Relire le résumé avec un plan B si le fichier manque, et produire le bulletin texte.

**À faire** :
1. Écris `lire_bulletin(chemin)` : renvoie le contenu du JSON, ou `None` si le fichier n'existe pas (`FileNotFoundError`).
2. Avec le bulletin relu, fabrique la liste `lignes` : une phrase par ville, `"Toulouse : 23.9 °C (75.0 °F), 108.3 mm"`.

**Résultat attendu** :
```
Toulouse : 23.9 °C (75.0 °F), 108.3 mm
Montpellier : 25.1 °C (77.2 °F), 119.9 mm
Nîmes : 26.7 °C (80.1 °F), 125.5 mm
Perpignan : 25.2 °C (77.4 °F), 136.3 mm
Albi : 22.7 °C (72.9 °F), 125.7 mm
Tarbes : 20.6 °C (69.1 °F), 74.5 mm
```

<details><summary>💡 Indice</summary>

Le `with` + `return json.load(fichier)` dans le `try` ; `return None` dans le `except FileNotFoundError:`. La phrase : une f-string avec `resume['ville']`, etc.

📖 Cours, sections 8 et 9.

</details>

In [ ]:
def lire_bulletin(chemin):
    """Contenu du JSON, ou None si le fichier n'existe pas."""
    ...

relu = lire_bulletin(SORTIES / "resume_septembre.json")
lignes = []
for resume in relu["villes"]:
    lignes.append(...)

for ligne in lignes:
    print(ligne)

In [ ]:
# Vérification
assert "lire_bulletin" in dir(), "Exécute ta cellule avec def lire_bulletin."
assert lire_bulletin(SORTIES / "absent.json") is None, "Pour un fichier absent, lire_bulletin doit renvoyer None (except FileNotFoundError)."
assert len(lignes) == 6, f"lignes doit contenir 6 phrases, tu en as {len(lignes)}."
assert lignes[0] == "Toulouse : 23.9 °C (75.0 °F), 108.3 mm", f"1re phrase inattendue : {lignes[0]!r}"
assert lignes[5] == "Tarbes : 20.6 °C (69.1 °F), 74.5 mm", f"Dernière phrase inattendue : {lignes[5]!r}"
print("✅ OK")

## 🏁 Ce que tu as construit

Ton premier **pipeline de données** : `meteo_brut.csv` (185 lignes abîmées) → nettoyage → `sorties/meteo_propre.csv`
(180 relevés) ; `meteo.csv` → résumés par ville → `sorties/resume_septembre.json`, relu et transformé en bulletin.
Ouvre les deux fichiers de `sorties/` dans VS Code pour les admirer.

**Au module 09**, tu arrêteras de recopier ta boîte à outils : tu la rangeras dans **ton propre module** `.py`
qu'on importe. Et tu iras chercher les **prévisions d'octobre** auprès d'une vraie API météo, pour les fusionner
avec cet historique.